# LR Calculation Parameter Selection

Self-contained RL workflow for learning when to `read` or `retrieve` a logistic-regression explanation, which drift-diffusion decision boundary to use, and which features to attend to.

This notebook now includes the training environment, data-loading helpers, training loop, evaluation helper, and plotting code directly.

Artifacts are saved under `outputs/rl_read_retrieve_policy/<run_name>/` with separate `models/`, `metrics/`, `plots/`, and `logs/` folders.

## 1. Setup

If imports fail, install the RL dependencies in your environment:

```bash
pip install gymnasium stable-baselines3
```

In [ ]:
from pathlib import Path
import sys
import importlib.util
import json
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "rl_agents":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import gymnasium as gym
    from gymnasium import spaces
    from stable_baselines3 import PPO
    from stable_baselines3.common.callbacks import EvalCallback
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.vec_env import DummyVecEnv
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook needs gymnasium and stable-baselines3. Install them before running training."
    ) from exc

from src.lr_memory import add_lr_calculation_to_memory, lr_calculation
from src.memory import CombinedMemory, DeclarativeMemory
from src.utils import AIDatasetLoader, LogisticRegressionInterpreter, filter_by_app_and_model

ROOT

## 2. Policy Environment and Helpers

The following cells define the configuration, dataset bundle loader, Gymnasium environment, training loop, evaluator, and plotting utilities used by the workflow below.

In [ ]:
from __future__ import annotations

ACCESS_MODES = {0: "retrieve", 1: "read"}


@dataclass(frozen=True)
class TrainingConfig:
    data_dir: str = str(ROOT / "datasets")
    output_root: str = str(ROOT / "outputs" / "rl_read_retrieve_policy")
    run_name: str | None = None
    total_timesteps: int = 100_000
    n_envs: int = 4
    instances_per_episode: int = 40
    max_features: int = 6
    explanation_shown_ratio: float = 0.5
    seed: int = 123
    learning_rate: float = 3e-4
    gamma: float = 0.85
    ent_coef: float = 0.01
    n_steps: int = 512
    batch_size: int = 256
    decision_boundary_bins: int = 5
    decision_boundary_min: float = 0.6
    decision_boundary_max: float = 1.8
    decision_noise_min: float = 0.3
    decision_noise_max: float = 1.2
    memory_recall_threshold_min: float = -1.0
    memory_recall_threshold_max: float = 2.0
    opportunity_cost_min: float = 0.0
    opportunity_cost_max: float = 0.03
    memory_recall_noise: float = 0.5
    retrieval_candidate_count: int = 3
    simulation_sample_count: int = 16
    read_seconds_per_item: float = 1.0
    mental_calculation_seconds: float = 0.0
    displayed_significant_figures: int = 2
    feature_cost: float = 0.002
    illegal_read_penalty: float = -1.0
    randomize_feature_order_per_episode: bool = True
    apps: tuple[str, ...] | None = None


@dataclass(frozen=True)
class DatasetBundle:
    app_id: str
    model_name: str
    loader: AIDatasetLoader
    lr_exp: LogisticRegressionInterpreter
    instance_ids: tuple[int, ...]


def _normalize_scalar(value: float, low: float, high: float) -> float:
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        return 0.0
    return float(np.clip((value - low) / (high - low), 0.0, 1.0))


def _safe_probabilities(probs: np.ndarray) -> np.ndarray:
    p = np.asarray(probs, dtype=float).copy()
    p[~np.isfinite(p)] = 0.0
    p[p < 0.0] = 0.0
    total = float(p.sum())
    if total <= 0.0:
        return np.full(len(p), 1.0 / max(1, len(p)), dtype=float)
    return p / total


def feature_base_index(feature_key: str) -> int:
    return int(feature_key.split("=")[0][1:])


def contribution_by_base_feature(lr_exp: LogisticRegressionInterpreter, x_raw: np.ndarray, max_features: int) -> np.ndarray:
    contributions = np.zeros(max_features, dtype=float)
    for feature_key, coefficient in lr_exp.coefficients.items():
        base_idx = feature_base_index(feature_key)
        if base_idx >= max_features or base_idx >= len(x_raw):
            continue
        if "=" in feature_key:
            base_key, category = feature_key.split("=")
            col = int(base_key[1:])
            value = 1.0 if int(x_raw[col]) == int(category) else 0.0
        else:
            value = float(x_raw[base_idx])
        contributions[base_idx] += float(coefficient) * value
    return contributions


def load_all_lr_bundles(data_dir: Path, apps: tuple[str, ...] | None = None) -> list[DatasetBundle]:
    values_df = pd.read_csv(data_dir / "values.csv")
    metadata_df = pd.read_csv(data_dir / "metadata.csv")
    prediction_df = pd.read_csv(data_dir / "none.csv")
    lr_df = pd.read_csv(data_dir / "logistic_regression.csv")

    base_loader = AIDatasetLoader(values_df, metadata_df, prediction_df)
    app_models = prediction_df[["appId", "modelName"]].drop_duplicates()
    if apps:
        app_models = app_models[app_models["appId"].isin(apps)]

    bundles: list[DatasetBundle] = []
    for row in app_models.itertuples(index=False):
        app_id = str(row.appId)
        model_name = str(row.modelName)
        try:
            loader = filter_by_app_and_model(base_loader, app_id, model_name)
            lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name)
        except Exception as exc:
            print(f"Skipping {app_id}/{model_name}: {exc}")
            continue
        feature_ids = set(loader.feature_values_df["instanceId"].dropna().astype(int).tolist())
        labeled_prediction_rows = loader.AI_predictions_df[loader.AI_predictions_df["pred"].notna()]
        prediction_ids = set(labeled_prediction_rows["instanceId"].dropna().astype(int).tolist())
        instance_ids = tuple(sorted(feature_ids & prediction_ids))
        if instance_ids:
            bundles.append(DatasetBundle(app_id, model_name, loader, lr_exp, instance_ids))
    return bundles


def make_memory(memory_recall_threshold: float, memory_recall_noise: float) -> CombinedMemory:
    dm = DeclarativeMemory(
        memory_recall_threshold=memory_recall_threshold,
        cue_association_strength=2.0,
        memory_mismatch_penalty=-2.0,
        memory_recall_noise=memory_recall_noise,
    )
    return CombinedMemory(dm, working_memory_capacity=7)


class ReadRetrievePolicyEnv(gym.Env):
    """
    PPO environment for selecting read/retrieve mode, DDM boundary, and attended features.

    Action:
      [mode_id, boundary_bin, feature_0, ..., feature_F]
      mode_id: 0=retrieve, 1=read
      boundary_bin: quantized value in config decision-boundary range
      feature_i: binary attend mask bit
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        bundles: list[DatasetBundle],
        config: TrainingConfig,
        *,
        training: bool = True,
        fixed_eval_params: dict[str, float] | None = None,
    ):
        super().__init__()
        if not bundles:
            raise ValueError("ReadRetrievePolicyEnv requires at least one DatasetBundle.")
        self.bundles = bundles
        self.config = config
        self.training = bool(training)
        self.fixed_eval_params = fixed_eval_params or {}

        self.action_space = spaces.MultiDiscrete(
            [2, config.decision_boundary_bins] + [2] * config.max_features
        )

        obs_dim = 9 + 3 * config.max_features
        self.observation_space = spaces.Box(
            low=np.zeros(obs_dim, dtype=np.float32),
            high=np.ones(obs_dim, dtype=np.float32),
            dtype=np.float32,
        )

        self.rng = np.random.default_rng(config.seed)
        self.bundle: DatasetBundle | None = None
        self.memory: CombinedMemory | None = None
        self.step_idx = 0
        self.current_params: dict[str, float] = {}
        self.explanation_schedule: np.ndarray | None = None
        self.X_raw: np.ndarray | None = None
        self.y: np.ndarray | None = None
        self.current_contrib = np.zeros(config.max_features, dtype=float)
        self.contrib_history: list[np.ndarray] = []
        self.mode_counts = np.zeros(2, dtype=float)
        self.mode_success = np.zeros(2, dtype=float)
        self.feature_order = np.arange(config.max_features, dtype=int)

    def _sample_param(self, key: str, low_attr: str, high_attr: str) -> float:
        if key in self.fixed_eval_params:
            return float(self.fixed_eval_params[key])
        low = float(getattr(self.config, low_attr))
        high = float(getattr(self.config, high_attr))
        if self.training:
            return float(self.rng.uniform(low, high))
        return 0.5 * (low + high)

    def _sample_episode_params(self) -> dict[str, float]:
        return {
            "decision_noise": self._sample_param("decision_noise", "decision_noise_min", "decision_noise_max"),
            "memory_recall_threshold": self._sample_param(
                "memory_recall_threshold",
                "memory_recall_threshold_min",
                "memory_recall_threshold_max",
            ),
            "opportunity_cost": self._sample_param(
                "opportunity_cost",
                "opportunity_cost_min",
                "opportunity_cost_max",
            ),
        }

    def _decision_boundary_from_bin(self, bin_id: int) -> float:
        bin_id = int(np.clip(bin_id, 0, self.config.decision_boundary_bins - 1))
        if self.config.decision_boundary_bins <= 1:
            return float(self.config.decision_boundary_min)
        frac = bin_id / max(1, self.config.decision_boundary_bins - 1)
        return float(
            self.config.decision_boundary_min
            + frac * (self.config.decision_boundary_max - self.config.decision_boundary_min)
        )

    def _build_explanation_schedule(self) -> np.ndarray:
        n = self.config.instances_per_episode
        n_readable = int(round(n * self.config.explanation_shown_ratio))
        flags = np.array([1] * n_readable + [0] * (n - n_readable), dtype=bool)
        self.rng.shuffle(flags)
        return flags

    def _sample_instances(self, bundle: DatasetBundle) -> tuple[np.ndarray, np.ndarray]:
        ids = np.asarray(bundle.instance_ids, dtype=int)
        replace = len(ids) < self.config.instances_per_episode
        chosen = self.rng.choice(ids, size=self.config.instances_per_episode, replace=replace).astype(int).tolist()
        raw_instances, labels = bundle.loader.load_instances(chosen, normalize=False)
        if any(label is None or pd.isna(label) for label in labels):
            missing = [instance_id for instance_id, label in zip(chosen, labels) if label is None or pd.isna(label)]
            raise ValueError(
                f"Sampled unlabeled instance(s) for {bundle.app_id}/{bundle.model_name}: {missing[:10]}"
            )
        return np.asarray(raw_instances, dtype=float), np.asarray(labels, dtype=int)

    def _initialize_memory(self) -> None:
        assert self.bundle is not None
        self.memory = make_memory(
            self.current_params["memory_recall_threshold"],
            self.config.memory_recall_noise,
        )
        add_lr_calculation_to_memory(
            self.bundle.lr_exp,
            self.memory,
            intercept_significant_figures=self.config.displayed_significant_figures,
            coefficient_significant_figures=self.config.displayed_significant_figures,
        )
        self.memory.tick(1.0)

    def _current_feature_stats(self) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        current_abs = np.abs(self.current_contrib)
        current_norm = current_abs / (float(current_abs.sum()) + 1e-9)
        if self.contrib_history:
            hist = np.vstack(self.contrib_history)
            means = np.abs(hist.mean(axis=0))
            stds = hist.std(axis=0)
        else:
            means = np.zeros(self.config.max_features, dtype=float)
            stds = np.zeros(self.config.max_features, dtype=float)
        mean_norm = means / (float(means.sum()) + 1e-9)
        std_norm = stds / (float(stds.sum()) + 1e-9)
        return current_norm, mean_norm, std_norm

    def _get_obs(self) -> np.ndarray:
        if self.step_idx >= self.config.instances_per_episode or self.explanation_schedule is None:
            return np.zeros(self.observation_space.shape, dtype=np.float32)
        current_norm, mean_norm, std_norm = self._current_feature_stats()
        if self.config.randomize_feature_order_per_episode:
            current_norm = current_norm[self.feature_order]
            mean_norm = mean_norm[self.feature_order]
            std_norm = std_norm[self.feature_order]
        obs = np.concatenate(
            [
                np.array(
                    [
                        _normalize_scalar(
                            self.current_params["decision_noise"],
                            self.config.decision_noise_min,
                            self.config.decision_noise_max,
                        ),
                        _normalize_scalar(
                            self.current_params["memory_recall_threshold"],
                            self.config.memory_recall_threshold_min,
                            self.config.memory_recall_threshold_max,
                        ),
                        _normalize_scalar(
                            self.current_params["opportunity_cost"],
                            self.config.opportunity_cost_min,
                            self.config.opportunity_cost_max,
                        ),
                        float(self.explanation_schedule[self.step_idx]),
                        float(self.step_idx / max(1, self.config.instances_per_episode)),
                        float(self.mode_counts[0] / max(1, self.step_idx)),
                        float(self.mode_counts[1] / max(1, self.step_idx)),
                        float(self.mode_success[0]),
                        float(self.mode_success[1]),
                    ],
                    dtype=np.float32,
                ),
                current_norm.astype(np.float32),
                mean_norm.astype(np.float32),
                std_norm.astype(np.float32),
            ]
        )
        return obs.astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.bundle = self.rng.choice(self.bundles)
        self.current_params = self._sample_episode_params()
        self._initialize_memory()
        self.explanation_schedule = self._build_explanation_schedule()
        self.X_raw, self.y = self._sample_instances(self.bundle)
        if self.config.randomize_feature_order_per_episode:
            self.feature_order = self.rng.permutation(self.config.max_features).astype(int)
        else:
            self.feature_order = np.arange(self.config.max_features, dtype=int)
        self.step_idx = 0
        self.contrib_history = []
        self.mode_counts[:] = 0.0
        self.mode_success[:] = 0.0
        self.current_contrib = contribution_by_base_feature(
            self.bundle.lr_exp,
            self.X_raw[self.step_idx],
            self.config.max_features,
        )
        return self._get_obs(), {}

    def step(self, action):
        assert self.bundle is not None
        assert self.memory is not None
        assert self.X_raw is not None
        assert self.y is not None
        assert self.explanation_schedule is not None

        action = np.asarray(action, dtype=int).reshape(-1)
        mode_id = int(action[0])
        boundary_bin = int(action[1])
        feature_slot_mask = action[2 : 2 + self.config.max_features].astype(int)
        chosen_mode = ACCESS_MODES.get(mode_id, "retrieve")
        with_explanation = bool(self.explanation_schedule[self.step_idx])
        decision_boundary = self._decision_boundary_from_bin(boundary_bin)
        selected_feature_slots = [int(i) for i, bit in enumerate(feature_slot_mask) if bit == 1]
        selected_features = [int(self.feature_order[i]) for i in selected_feature_slots]
        feature_mask = np.zeros(self.config.max_features, dtype=int)
        for actual_idx in selected_features:
            if 0 <= actual_idx < self.config.max_features:
                feature_mask[actual_idx] = 1

        x_raw = self.X_raw[self.step_idx]
        y_true = int(self.y[self.step_idx])

        if chosen_mode == "read" and not with_explanation:
            prob_correct = 0.0
            pred_time = 0.0
            reward = float(self.config.illegal_read_penalty)
            illegal_action = True
        else:
            probs, pred_time, _info = lr_calculation(
                x_raw,
                self.memory,
                self.bundle.lr_exp,
                explanation_access_mode=chosen_mode,
                displayed_significant_figures=self.config.displayed_significant_figures,
                read_seconds_per_item=self.config.read_seconds_per_item,
                mental_calculation_seconds=self.config.mental_calculation_seconds,
                decision_boundary=decision_boundary,
                decision_noise=self.current_params["decision_noise"],
                selected_feature_indices=selected_features,
                simulation_sample_count=self.config.simulation_sample_count,
                retrieval_candidate_count=self.config.retrieval_candidate_count,
            )
            probs = _safe_probabilities(probs)
            prob_correct = float(probs[y_true]) if y_true < len(probs) else 0.0
            reward = (
                prob_correct
                - self.current_params["opportunity_cost"] * float(pred_time)
                - self.config.feature_cost * float(len(selected_features))
            )
            illegal_action = False

        mode_index = 0 if chosen_mode == "retrieve" else 1
        self.mode_counts[mode_index] += 1.0
        n = self.mode_counts[mode_index]
        self.mode_success[mode_index] = (
            (self.mode_success[mode_index] * (n - 1.0)) + float(prob_correct >= 0.5)
        ) / max(n, 1.0)

        info = {
            "app_id": self.bundle.app_id,
            "model_name": self.bundle.model_name,
            "chosen_mode": chosen_mode,
            "with_explanation": with_explanation,
            "decision_boundary": decision_boundary,
            "decision_boundary_bin": boundary_bin,
            "decision_noise": self.current_params["decision_noise"],
            "memory_recall_threshold": self.current_params["memory_recall_threshold"],
            "opportunity_cost": self.current_params["opportunity_cost"],
            "prob_correct": prob_correct,
            "pred_time": float(pred_time),
            "reward_without_illegal_penalty": float(
                prob_correct
                - self.current_params["opportunity_cost"] * float(pred_time)
                - self.config.feature_cost * float(len(selected_features))
            ),
            "n_selected_features": len(selected_features),
            "selected_features": selected_features,
            "selected_feature_slots": selected_feature_slots,
            "feature_mask": feature_mask.astype(int).tolist(),
            "feature_slot_mask": feature_slot_mask.astype(int).tolist(),
            "feature_order": self.feature_order.astype(int).tolist(),
            "illegal_action": illegal_action,
        }

        self.contrib_history.append(self.current_contrib.copy())
        self.step_idx += 1
        terminated = False
        truncated = self.step_idx >= self.config.instances_per_episode
        if not truncated:
            self.current_contrib = contribution_by_base_feature(
                self.bundle.lr_exp,
                self.X_raw[self.step_idx],
                self.config.max_features,
            )
        return self._get_obs(), float(reward), terminated, truncated, info

In [ ]:
from __future__ import annotations

def make_run_dir(config: TrainingConfig) -> Path:
    run_name = config.run_name or datetime.now().strftime("read_retrieve_policy_%Y%m%d_%H%M%S")
    run_dir = Path(config.output_root) / run_name
    for subdir in ["models", "logs", "plots", "metrics"]:
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)
    return run_dir


def make_vec_env(bundles: list[DatasetBundle], config: TrainingConfig, n_envs: int, training: bool):
    def factory(rank: int):
        def _init():
            env = ReadRetrievePolicyEnv(bundles, config, training=training)
            env.reset(seed=config.seed + rank)
            return Monitor(env)
        return _init

    return DummyVecEnv([factory(i) for i in range(n_envs)])


def train_policy(config: TrainingConfig) -> tuple[PPO, Path, list[DatasetBundle]]:
    run_dir = make_run_dir(config)
    bundles = load_all_lr_bundles(Path(config.data_dir), apps=config.apps)
    if not bundles:
        raise RuntimeError("No dataset/model bundles were loaded.")

    (run_dir / "config.json").write_text(json.dumps(asdict(config), indent=2), encoding="utf-8")
    bundle_table = pd.DataFrame(
        [{"app_id": b.app_id, "model_name": b.model_name, "n_instances": len(b.instance_ids)} for b in bundles]
    )
    bundle_table.to_csv(run_dir / "metrics" / "bundles.csv", index=False)

    train_env = make_vec_env(bundles, config, config.n_envs, training=True)
    eval_env = make_vec_env(bundles, config, 1, training=False)

    model = PPO(
        "MlpPolicy",
        train_env,
        learning_rate=config.learning_rate,
        gamma=config.gamma,
        ent_coef=config.ent_coef,
        n_steps=config.n_steps,
        batch_size=config.batch_size,
        verbose=1,
        seed=config.seed,
        tensorboard_log=str(run_dir / "logs") if importlib.util.find_spec("tensorboard") else None,
        device="cpu",
    )
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(run_dir / "models"),
        log_path=str(run_dir / "metrics"),
        eval_freq=max(config.n_steps, 1),
        deterministic=True,
    )

    started = time.time()
    model.learn(total_timesteps=config.total_timesteps, callback=eval_callback, progress_bar=False)
    elapsed = time.time() - started

    model_path = run_dir / "models" / "final_model.zip"
    model.save(model_path)
    (run_dir / "metrics" / "training_summary.json").write_text(
        json.dumps({"elapsed_seconds": elapsed, "model_path": str(model_path)}, indent=2),
        encoding="utf-8",
    )
    train_env.close()
    eval_env.close()
    return model, run_dir, bundles


def evaluate_policy(
    model: PPO,
    bundles: list[DatasetBundle],
    config: TrainingConfig,
    *,
    n_episodes: int = 20,
    deterministic: bool = True,
    fixed_eval_params: dict[str, float] | None = None,
    sample_parameters: bool = True,
) -> pd.DataFrame:
    """Evaluate the learned policy and return one row per decision.

    By default this samples decision noise, retrieval threshold, and opportunity cost
    across episodes, matching the parameter variation seen during training. Set
    sample_parameters=False for a deterministic midpoint evaluation, or pass
    fixed_eval_params to pin one or more parameters to specific values.
    """
    env = ReadRetrievePolicyEnv(
        bundles,
        config,
        training=sample_parameters,
        fixed_eval_params=fixed_eval_params,
    )
    rows: list[dict[str, Any]] = []
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=config.seed + 10_000 + episode)
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            done = bool(terminated or truncated)
            rows.append(
                {
                    "episode": episode,
                    "step": env.step_idx - 1,
                    "reward": float(reward),
                    **{k: v for k, v in info.items() if k not in {"selected_features", "selected_feature_slots", "feature_mask", "feature_slot_mask", "feature_order"}},
                    "selected_features": ",".join(str(i) for i in info.get("selected_features", [])),
                    "selected_feature_slots": ",".join(str(i) for i in info.get("selected_feature_slots", [])),
                    "feature_order": ",".join(str(i) for i in info.get("feature_order", [])),
                    **{
                        f"feature_{i}_selected": int(bit)
                        for i, bit in enumerate(info.get("feature_mask", []))
                    },
                    **{
                        f"slot_{i}_selected": int(bit)
                        for i, bit in enumerate(info.get("feature_slot_mask", []))
                    },
                }
            )
    env.close()
    return pd.DataFrame(rows)


def _display_or_show(fig):
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        fig.show()


def _with_dataset_label(df: pd.DataFrame) -> pd.DataFrame:
    labeled = df.copy()
    labeled["dataset"] = labeled["app_id"].astype(str) + "/" + labeled["model_name"].astype(str)
    return labeled


def _plot_dataset_metric_lines(
    ax,
    eval_df: pd.DataFrame,
    x_col: str,
    y_col: str,
    y_label: str,
    title: str,
    *,
    n_bins: int = 8,
    scatter: bool = True,
) -> None:
    plot_df = _with_dataset_label(eval_df)
    datasets = sorted(plot_df["dataset"].unique())
    cmap = plt.get_cmap("tab20", max(1, len(datasets)))
    x_min = float(plot_df[x_col].min())
    x_max = float(plot_df[x_col].max())

    for idx, dataset in enumerate(datasets):
        dataset_df = plot_df[plot_df["dataset"] == dataset].copy()
        color = cmap(idx)
        if scatter:
            ax.scatter(dataset_df[x_col], dataset_df[y_col], s=12, alpha=0.12, color=color)

        if x_max > x_min and dataset_df[x_col].nunique() > 1:
            dataset_df["x_bin"] = pd.cut(
                dataset_df[x_col],
                bins=np.linspace(x_min, x_max, n_bins + 1),
                include_lowest=True,
            )
            binned = dataset_df.groupby("x_bin", observed=False).agg(
                x_mean=(x_col, "mean"),
                y_mean=(y_col, "mean"),
            ).dropna()
            ax.plot(binned["x_mean"], binned["y_mean"], linewidth=2.0, marker="o", markersize=3, color=color, label=dataset)
        else:
            ax.axhline(float(dataset_df[y_col].mean()), linewidth=2.0, color=color, label=dataset)

    ax.set_title(title)
    ax.set_xlabel(x_col.replace("_", " ").title())
    ax.set_ylabel(y_label)
    ax.grid(alpha=0.2)


def _save_display_close(fig, path: Path, saved_paths: list[Path], show: bool) -> None:
    fig.savefig(path, dpi=160, bbox_inches="tight")
    saved_paths.append(path)
    if show:
        _display_or_show(fig)
    plt.close(fig)


def plot_evaluation_summary(eval_df: pd.DataFrame, out_dir: Path, max_features: int, *, show: bool = True) -> list[Path]:
    plot_dir = out_dir / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)
    saved_paths: list[Path] = []
    if eval_df.empty:
        return saved_paths

    labeled_df = _with_dataset_label(eval_df)
    feature_cols = [f"feature_{i}_selected" for i in range(max_features) if f"feature_{i}_selected" in eval_df]

    # Boundary first: this is the learned parameter we care about most.
    fig, axes = plt.subplots(2, 1, figsize=(12, 11), sharex=False)
    _plot_dataset_metric_lines(
        axes[0],
        labeled_df,
        "opportunity_cost",
        "decision_boundary",
        "Chosen boundary",
        "Decision boundary vs opportunity cost by dataset",
    )
    _plot_dataset_metric_lines(
        axes[1],
        labeled_df,
        "decision_noise",
        "decision_boundary",
        "Chosen boundary",
        "Decision boundary vs decision noise by dataset",
    )
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    fig.tight_layout(rect=(0, 0, 0.78, 1))
    _save_display_close(fig, plot_dir / "boundary_analysis_by_dataset.png", saved_paths, show)

    # Feature-selection analysis: how many features, plus which features.
    fig, axes = plt.subplots(2, 1, figsize=(12, 12), gridspec_kw={"height_ratios": [1.0, 1.2]})
    _plot_dataset_metric_lines(
        axes[0],
        labeled_df,
        "opportunity_cost",
        "n_selected_features",
        "Selected feature count",
        "Number of selected features vs opportunity cost by dataset",
    )

    if feature_cols:
        feature_rates = labeled_df.groupby("dataset")[feature_cols].mean().sort_index()
        image = axes[1].imshow(feature_rates.to_numpy(), aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
        axes[1].set_yticks(np.arange(len(feature_rates.index)))
        axes[1].set_yticklabels(feature_rates.index)
        axes[1].set_xticks(np.arange(len(feature_cols)))
        axes[1].set_xticklabels([c.replace("feature_", "f").replace("_selected", "") for c in feature_cols])
        axes[1].set_title("Actual feature selection rate by dataset")
        axes[1].set_xlabel("Actual feature index")
        axes[1].set_ylabel("Dataset/model")
        fig.colorbar(image, ax=axes[1], fraction=0.025, pad=0.02, label="Selection rate")
    else:
        axes[1].axis("off")
    fig.tight_layout()
    _save_display_close(fig, plot_dir / "feature_selection_analysis_by_dataset.png", saved_paths, show)

    # Outcomes are still useful, but keep probability, time, and reward separate.
    fig, axes = plt.subplots(3, 1, figsize=(12, 14), sharex=True)
    _plot_dataset_metric_lines(
        axes[0],
        labeled_df,
        "opportunity_cost",
        "prob_correct",
        "Probability assigned to true label",
        "Correct probability vs opportunity cost by dataset",
    )
    _plot_dataset_metric_lines(
        axes[1],
        labeled_df,
        "opportunity_cost",
        "pred_time",
        "Predicted decision time",
        "Decision time vs opportunity cost by dataset",
    )
    _plot_dataset_metric_lines(
        axes[2],
        labeled_df,
        "opportunity_cost",
        "reward",
        "Reward",
        "Reward vs opportunity cost by dataset",
    )
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    fig.tight_layout(rect=(0, 0, 0.78, 1))
    _save_display_close(fig, plot_dir / "outcomes_by_dataset.png", saved_paths, show)

    return saved_paths


def _display_dataframe(df: pd.DataFrame) -> None:
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string(index=False))


def feature_name_lookup(bundles: list[DatasetBundle], max_features: int) -> dict[str, dict[int, str]]:
    lookup: dict[str, dict[int, str]] = {}
    for bundle in bundles:
        dataset = f"{bundle.app_id}/{bundle.model_name}"
        names = {}
        for i in range(max_features):
            key = f"a{i}"
            try:
                names[i] = str(bundle.lr_exp._format_feature(key))
            except Exception:
                names[i] = key
        lookup[dataset] = names
    return lookup


def actual_feature_focus_table(eval_df: pd.DataFrame, bundles: list[DatasetBundle], max_features: int) -> pd.DataFrame:
    """Return actual, unshuffled feature selection rates by dataset/model."""
    if eval_df.empty:
        return pd.DataFrame()
    labeled = _with_dataset_label(eval_df)
    names = feature_name_lookup(bundles, max_features)
    rows: list[dict[str, Any]] = []
    for dataset, dataset_df in labeled.groupby("dataset"):
        for i in range(max_features):
            col = f"feature_{i}_selected"
            if col not in dataset_df:
                continue
            rows.append(
                {
                    "dataset": dataset,
                    "feature_index": i,
                    "feature_name": names.get(dataset, {}).get(i, f"a{i}"),
                    "selection_rate": float(dataset_df[col].mean()),
                    "n_decisions": int(len(dataset_df)),
                }
            )
    return pd.DataFrame(rows).sort_values(["dataset", "selection_rate"], ascending=[True, False])


def print_top_actual_feature_focus(eval_df: pd.DataFrame, bundles: list[DatasetBundle], max_features: int, *, top_n: int = 6) -> pd.DataFrame:
    focus = actual_feature_focus_table(eval_df, bundles, max_features)
    if focus.empty:
        print("No feature-selection columns found.")
        return focus
    print("Actual feature focus after undoing per-episode feature-slot randomization:")
    top = focus.groupby("dataset", group_keys=False).head(top_n).reset_index(drop=True)
    _display_dataframe(top)
    return focus


def summarize_parameter_variation(
    eval_df: pd.DataFrame,
    parameter_col: str,
    *,
    n_bins: int = 6,
    max_features: int = 6,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Summarize policy choices/outcomes and feature rates across bins of one varied parameter."""
    if eval_df.empty or parameter_col not in eval_df:
        return pd.DataFrame(), pd.DataFrame()

    df = eval_df.copy()
    df["hard_correct"] = (df["prob_correct"] >= 0.5).astype(float)
    df["read_rate"] = (df["chosen_mode"] == "read").astype(float)
    df["retrieve_rate"] = (df["chosen_mode"] == "retrieve").astype(float)
    df["illegal_rate"] = df["illegal_action"].astype(float) if "illegal_action" in df else 0.0

    unique_values = df[parameter_col].dropna().nunique()
    if unique_values <= 1:
        df["parameter_bin"] = "all"
    else:
        bins = min(int(n_bins), int(unique_values))
        df["parameter_bin"] = pd.qcut(df[parameter_col], q=bins, duplicates="drop")

    summary = (
        df.groupby("parameter_bin", observed=False)
        .agg(
            n_decisions=("reward", "size"),
            parameter_min=(parameter_col, "min"),
            parameter_mean=(parameter_col, "mean"),
            parameter_max=(parameter_col, "max"),
            mean_selected_features=("n_selected_features", "mean"),
            mean_decision_boundary=("decision_boundary", "mean"),
            mean_prob_correct=("prob_correct", "mean"),
            hard_accuracy=("hard_correct", "mean"),
            mean_pred_time=("pred_time", "mean"),
            mean_reward=("reward", "mean"),
            read_rate=("read_rate", "mean"),
            retrieve_rate=("retrieve_rate", "mean"),
            illegal_rate=("illegal_rate", "mean"),
        )
        .reset_index()
    )

    feature_cols = [f"feature_{i}_selected" for i in range(max_features) if f"feature_{i}_selected" in df]
    if feature_cols:
        feature_rates = df.groupby("parameter_bin", observed=False)[feature_cols].mean().reset_index()
        feature_rates = feature_rates.rename(columns={f"feature_{i}_selected": f"actual_feature_{i}_rate" for i in range(max_features)})
    else:
        feature_rates = pd.DataFrame()
    return summary, feature_rates


def print_parameter_variation(
    eval_df: pd.DataFrame,
    parameter_col: str,
    *,
    n_bins: int = 6,
    max_features: int = 6,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary, feature_rates = summarize_parameter_variation(
        eval_df,
        parameter_col,
        n_bins=n_bins,
        max_features=max_features,
    )
    print(f"Policy/outcome variation by {parameter_col}:")
    _display_dataframe(summary)
    if not feature_rates.empty:
        print(f"Actual feature selection rates by {parameter_col}:")
        _display_dataframe(feature_rates)
    return summary, feature_rates


def print_all_parameter_variations(
    eval_df: pd.DataFrame,
    *,
    parameter_cols: tuple[str, ...] = ("memory_recall_threshold", "decision_noise", "opportunity_cost"),
    n_bins: int = 6,
    max_features: int = 6,
) -> dict[str, tuple[pd.DataFrame, pd.DataFrame]]:
    results = {}
    for parameter_col in parameter_cols:
        if parameter_col not in eval_df:
            continue
        results[parameter_col] = print_parameter_variation(
            eval_df,
            parameter_col,
            n_bins=n_bins,
            max_features=max_features,
        )
    return results


def _sample_eval_params(config: TrainingConfig, rng: np.random.Generator, sample_parameters: bool) -> dict[str, float]:
    if sample_parameters:
        return {
            "decision_noise": float(rng.uniform(config.decision_noise_min, config.decision_noise_max)),
            "memory_recall_threshold": float(rng.uniform(config.memory_recall_threshold_min, config.memory_recall_threshold_max)),
            "opportunity_cost": float(rng.uniform(config.opportunity_cost_min, config.opportunity_cost_max)),
        }
    return {
        "decision_noise": 0.5 * (config.decision_noise_min + config.decision_noise_max),
        "memory_recall_threshold": 0.5 * (config.memory_recall_threshold_min + config.memory_recall_threshold_max),
        "opportunity_cost": 0.5 * (config.opportunity_cost_min + config.opportunity_cost_max),
    }


def evaluate_fixed_feature_boundary_policy(
    bundles: list[DatasetBundle],
    config: TrainingConfig,
    *,
    decision_boundary: float = 1.0,
    n_per_dataset: int = 40,
    modes: tuple[str, ...] = ("retrieve", "read"),
    sample_parameters: bool = True,
) -> pd.DataFrame:
    """Run a direct what-if analysis with all features selected and a fixed boundary.

    This bypasses the learned policy action and asks: if the calculation always used
    every feature and decision boundary = 1.0, what probability/time/reward would it
    produce for each dataset? The `read` rows assume the explanation is available.
    """
    rng = np.random.default_rng(config.seed + 20_000)
    selected_features = list(range(config.max_features))
    rows: list[dict[str, Any]] = []

    for bundle in bundles:
        ids = np.asarray(bundle.instance_ids, dtype=int)
        if len(ids) == 0:
            continue
        replace = len(ids) < n_per_dataset
        chosen = rng.choice(ids, size=n_per_dataset, replace=replace).astype(int).tolist()
        X_raw, y = bundle.loader.load_instances(chosen, normalize=False)

        for row_idx, (instance_id, x_raw, y_true) in enumerate(zip(chosen, X_raw, y)):
            if y_true is None or pd.isna(y_true):
                continue
            params = _sample_eval_params(config, rng, sample_parameters)
            for mode in modes:
                memory = make_memory(params["memory_recall_threshold"], config.memory_recall_noise)
                add_lr_calculation_to_memory(
                    bundle.lr_exp,
                    memory,
                    intercept_significant_figures=config.displayed_significant_figures,
                    coefficient_significant_figures=config.displayed_significant_figures,
                )
                memory.tick(1.0)
                probs, pred_time, _info = lr_calculation(
                    np.asarray(x_raw, dtype=float),
                    memory,
                    bundle.lr_exp,
                    explanation_access_mode=mode,
                    displayed_significant_figures=config.displayed_significant_figures,
                    read_seconds_per_item=config.read_seconds_per_item,
                    mental_calculation_seconds=config.mental_calculation_seconds,
                    decision_boundary=decision_boundary,
                    decision_noise=params["decision_noise"],
                    selected_feature_indices=selected_features,
                    simulation_sample_count=config.simulation_sample_count,
                    retrieval_candidate_count=config.retrieval_candidate_count,
                )
                probs = _safe_probabilities(probs)
                y_idx = int(y_true)
                prob_correct = float(probs[y_idx]) if y_idx < len(probs) else 0.0
                reward = (
                    prob_correct
                    - params["opportunity_cost"] * float(pred_time)
                    - config.feature_cost * float(len(selected_features))
                )
                rows.append(
                    {
                        "app_id": bundle.app_id,
                        "model_name": bundle.model_name,
                        "dataset": f"{bundle.app_id}/{bundle.model_name}",
                        "instance_id": int(instance_id),
                        "row_idx": int(row_idx),
                        "calculation_mode": mode,
                        "decision_boundary": float(decision_boundary),
                        "n_selected_features": len(selected_features),
                        "prob_correct": prob_correct,
                        "pred_time": float(pred_time),
                        "reward": float(reward),
                        **params,
                    }
                )
    return pd.DataFrame(rows)


def summarize_fixed_feature_boundary_stats(fixed_df: pd.DataFrame) -> pd.DataFrame:
    if fixed_df.empty:
        return fixed_df
    return (
        fixed_df.groupby(["dataset", "calculation_mode"], as_index=False)
        .agg(
            n_trials=("prob_correct", "size"),
            mean_prob_correct=("prob_correct", "mean"),
            mean_pred_time=("pred_time", "mean"),
            mean_reward=("reward", "mean"),
            sd_prob_correct=("prob_correct", "std"),
            sd_pred_time=("pred_time", "std"),
            sd_reward=("reward", "std"),
        )
        .sort_values(["dataset", "calculation_mode"])
    )


def plot_fixed_feature_boundary_stats(fixed_df: pd.DataFrame, *, show: bool = True) -> list[Path]:
    saved_paths: list[Path] = []
    if fixed_df.empty:
        return saved_paths

    summary = summarize_fixed_feature_boundary_stats(fixed_df)
    metrics = [
        ("mean_prob_correct", "Mean probability assigned to true label", "All features, boundary = 1.0: correct probability"),
        ("mean_pred_time", "Mean predicted decision time", "All features, boundary = 1.0: time"),
        ("mean_reward", "Mean reward", "All features, boundary = 1.0: reward"),
    ]
    datasets = sorted(summary["dataset"].unique())
    modes = list(summary["calculation_mode"].drop_duplicates())
    y = np.arange(len(datasets))
    height = 0.8 / max(1, len(modes))
    colors = {"retrieve": "#27AE60", "read": "#2F80ED"}

    fig, axes = plt.subplots(1, 3, figsize=(17, max(5, 0.38 * len(datasets) + 2)), sharey=True)
    for ax, (metric, x_label, title) in zip(axes, metrics):
        for mode_idx, mode in enumerate(modes):
            mode_summary = summary[summary["calculation_mode"] == mode].set_index("dataset").reindex(datasets)
            offset = (mode_idx - (len(modes) - 1) / 2) * height
            ax.barh(y + offset, mode_summary[metric], height=height, label=mode, color=colors.get(mode, None), alpha=0.85)
        ax.set_title(title)
        ax.set_xlabel(x_label)
        ax.grid(axis="x", alpha=0.2)
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(datasets)
    axes[0].invert_yaxis()
    axes[-1].legend(title="Calculation mode", frameon=False, loc="lower right")
    fig.tight_layout()

    if show:
        _display_or_show(fig)
    else:
        plt.close(fig)

    return saved_paths

def run_training_workflow(config: TrainingConfig, eval_episodes: int = 20) -> tuple[PPO, Path, pd.DataFrame]:
    model, run_dir, bundles = train_policy(config)
    eval_df = evaluate_policy(model, bundles, config, n_episodes=eval_episodes)
    eval_csv = run_dir / "metrics" / "evaluation.csv"
    eval_df.to_csv(eval_csv, index=False)
    plot_evaluation_summary(eval_df, run_dir, config.max_features, show=True)
    print(f"Saved run artifacts to {run_dir}")
    return model, run_dir, eval_df

## 3. Configure Training

The default config trains over every app/model pair available in `datasets/none.csv` that also has a logistic-regression explanation in `datasets/logistic_regression.csv`.

For a smoke test, use a small `total_timesteps` such as `5_000`. For a real run, increase it to `100_000` or more.

In [ ]:
config = TrainingConfig(
    data_dir=str(ROOT / "datasets"),
    output_root=str(ROOT / "outputs" / "rl_read_retrieve_policy"),
    run_name="lr_calculation_parameter_selection_demo",
    total_timesteps=5e5,
    n_envs=4,
    instances_per_episode=40,
    max_features=6,
    explanation_shown_ratio=0.5,
    decision_boundary_bins=5,
    decision_boundary_min=0.6,
    decision_boundary_max=1.8,
    decision_noise_min=0.3,
    decision_noise_max=0.5,
    memory_recall_threshold_min=-1.0,
    memory_recall_threshold_max=2.0,
    opportunity_cost_min=0.0,
    opportunity_cost_max=0.02,
    memory_recall_noise=0.5,
    retrieval_candidate_count=3,
    simulation_sample_count=16,
    seed=123,
)

config

## 4. Check Available Datasets

In [ ]:
bundles = load_all_lr_bundles(Path(config.data_dir), apps=config.apps)
[(b.app_id, b.model_name, len(b.instance_ids)) for b in bundles]

In [ ]:
# Inspect one LR calculation before any RL training runs.
from IPython.display import display

demo_bundle = next((b for b in bundles if b.app_id == "mushrooms"), bundles[0])
demo_instance_id = int(demo_bundle.instance_ids[0])
demo_X, demo_y = demo_bundle.loader.load_instances([demo_instance_id], normalize=False)
demo_x = np.asarray(demo_X[0], dtype=float)
demo_label = None if demo_y[0] is None or pd.isna(demo_y[0]) else int(demo_y[0])
demo_selected_features = list(range(min(config.max_features, len(demo_x))))

demo_memory = make_memory(memory_recall_threshold=0.0, memory_recall_noise=config.memory_recall_noise)
add_lr_calculation_to_memory(
    demo_bundle.lr_exp,
    demo_memory,
    intercept_significant_figures=config.displayed_significant_figures,
    coefficient_significant_figures=config.displayed_significant_figures,
)
demo_memory.tick(1.0)

demo_probs, demo_time, demo_info = lr_calculation(
    demo_x,
    demo_memory,
    demo_bundle.lr_exp,
    explanation_access_mode="read",
    displayed_significant_figures=config.displayed_significant_figures,
    read_seconds_per_item=config.read_seconds_per_item,
    mental_calculation_seconds=config.mental_calculation_seconds,
    decision_boundary=1.0,
    decision_noise=0.4,
    selected_feature_indices=demo_selected_features,
    simulation_sample_count=config.simulation_sample_count,
    retrieval_candidate_count=config.retrieval_candidate_count,
    verbose=True,
)

demo_calculation_df = pd.DataFrame(demo_info["calculation_rows"])
demo_calculation_df.insert(
    1,
    "feature_name",
    demo_calculation_df["feature_key"].map(
        lambda key: "Intercept" if key == "intercept" else demo_bundle.lr_exp._format_feature(key)
    ),
)
demo_calculation_df["calculation"] = demo_calculation_df.apply(
    lambda row: f"{row.coefficient_used:.4g}"
    if row.feature_key == "intercept"
    else f"{row.coefficient_used:.4g} * {row.value_used:.4g}",
    axis=1,
)
demo_summary = pd.Series(
    {
        "dataset": f"{demo_bundle.app_id}/{demo_bundle.model_name}",
        "instance_id": demo_instance_id,
        "ai_prediction_label": demo_label,
        "selected_feature_indices": demo_selected_features,
        "rounded_lr_sum": demo_info["sum"],
        "ddm_evidence": demo_info["evidence"],
        "p_class_0": float(demo_probs[0]),
        "p_class_1": float(demo_probs[1]),
        "predicted_time_seconds": demo_time,
        "ops_count": demo_info["ops_count"],
    }
)

display(demo_summary.to_frame("value"))
display(demo_calculation_df)
assert np.isclose(demo_calculation_df["contribution"].sum(), demo_info["sum"])


## 5. Train and Save the Model

This creates a run folder with:

- `models/final_model.zip`
- `models/best_model.zip` from the evaluation callback
- `metrics/config.json`, `metrics/bundles.csv`, `metrics/evaluation.csv`
- `plots/*.png`
- `logs/` for TensorBoard when the optional `tensorboard` package is installed

In [ ]:
model, run_dir, bundles = train_policy(config)
run_dir

## 6. Evaluate the Learned Policy

Evaluation now samples decision noise, retrieval threshold, and opportunity cost across episodes by default. Previously, evaluation used midpoint values, which is why opportunity cost appeared stuck at `0.015`.

In [ ]:
eval_df = evaluate_policy(
    model,
    bundles,
    config,
    n_episodes=200,
    deterministic=True,
    sample_parameters=True,
)
eval_path = run_dir / "metrics" / "evaluation.csv"
eval_df.to_csv(eval_path, index=False)

parameter_ranges = eval_df[["decision_noise", "memory_recall_threshold", "opportunity_cost"]].agg(["min", "mean", "max"])
display(parameter_ranges)
parameter_variation_tables = print_all_parameter_variations(eval_df, n_bins=6, max_features=config.max_features)
actual_feature_focus = print_top_actual_feature_focus(eval_df, bundles, config.max_features, top_n=config.max_features)
eval_df.head()

In [ ]:
eval_summary = (
    _with_dataset_label(eval_df)
    .groupby("dataset", as_index=False)
    .agg(
        mean_boundary=("decision_boundary", "mean"),
        mean_selected_features=("n_selected_features", "mean"),
        mean_prob_correct=("prob_correct", "mean"),
        mean_pred_time=("pred_time", "mean"),
        mean_reward=("reward", "mean"),
        min_opportunity_cost=("opportunity_cost", "min"),
        max_opportunity_cost=("opportunity_cost", "max"),
    )
    .sort_values("dataset")
)
eval_summary

## 7. Boundary and Feature Analysis

The main plots now prioritize the learned decision boundary and selected-feature behavior. Lines are separated by dataset/model so aggregate trends do not hide dataset-specific patterns.

In [ ]:
plot_paths = plot_evaluation_summary(eval_df, run_dir, config.max_features, show=True)
plot_paths

## 8. Fixed All-Features Boundary Baseline

This cell directly asks: if every policy-selectable feature is selected and the decision boundary is fixed at `1.0`, what time, probability correct, and reward do we get by dataset? The `read` rows assume the explanation is available.

In [ ]:
config = TrainingConfig(
    data_dir=str(ROOT / "datasets"),
    output_root=str(ROOT / "outputs" / "rl_read_retrieve_policy"),
    run_name="lr_calculation_parameter_selection_demo",
    total_timesteps=100_000,
    n_envs=4,
    instances_per_episode=40,
    max_features=6,
    explanation_shown_ratio=0.5,
    decision_boundary_bins=5,
    decision_boundary_min=1.7,
    decision_boundary_max=1.8,
    decision_noise_min=0.3,
    decision_noise_max=0.5,
    memory_recall_threshold_min=-1.0,
    memory_recall_threshold_max=-1.0,
    opportunity_cost_min=0.0,
    opportunity_cost_max=0.01,
    memory_recall_noise=0.5,
    retrieval_candidate_count=3,
    simulation_sample_count=16,
    seed=123,
)

In [ ]:
fixed_all_features_df = evaluate_fixed_feature_boundary_policy(
    bundles,
    config,
    decision_boundary=1.0,
    n_per_dataset=40,
    modes=("retrieve", "read"),
    sample_parameters=True,
)
# fixed_all_features_path = run_dir / "metrics" / "fixed_all_features_boundary_1.csv"
# fixed_all_features_df.to_csv(fixed_all_features_path, index=False)

fixed_all_features_summary = summarize_fixed_feature_boundary_stats(fixed_all_features_df)
display(fixed_all_features_summary)

fixed_plot_paths = plot_fixed_feature_boundary_stats(fixed_all_features_df, show=True)
fixed_plot_paths

## 9. Optional One-Cell Workflow

Use this when you want training, evaluation, CSV saving, and graph generation in a single call.

In [ ]:
# model, run_dir, eval_df = run_training_workflow(config, eval_episodes=30)